# Your first agent with a framework

**Goal:** build the same kind of agent as [agent_from_scratch.ipynb](agent_from_scratch.ipynb), with less code to manage the loop. Then add conversation memory to make it more useful.

Our agent answers questions about **Philadelphia and other US cities** using two Python tools: addition and city distance.

In this demo we use **LangChain's `create_agent`**, which runs on **LangGraph**. Start with this ready-made agent; you can learn to customize the graph later.

## 1. Why use a framework?

In the first notebook, you wrote these steps yourself:

```text
Ask the model → check for a tool request → run the tool
      ↑                                      │
      └──────── send back the result ──────────┘
Stop when the model gives its answer.
```

A framework provides this reusable agent loop. **You still write the tools and instructions.** It handles calling the model again, running requested tools, and adding their results to the conversation.

| Task | Without a framework | With this framework |
| --- | --- | --- |
| Describe tool inputs | Write JSON schemas yourself. | `@tool` builds schemas from type hints and docstrings. |
| Run the agent loop | Write and maintain the loop and tool dispatch. | `create_agent` supplies the loop. |
| Remember earlier questions | Save and resend conversation history yourself. | Add a checkpointer and use a conversation ID. |
| Build a more complete agent | Add more supporting code as features grow. | Reuse tools, memory, and graph features. |

**Why is this useful?** You can spend more time developing useful tools and checking answers, and less time maintaining the plumbing. For a very small agent, plain Python is still a good choice. A framework saves development work; it does not make the model automatically more accurate.

**Two names to remember:** LangChain gives us convenient model, tool, and agent interfaces. LangGraph runs the workflow and manages its state. A graph describes which step runs next; a fixed chain follows a preset sequence, while this agent can repeat tool steps as needed.

Sources: [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview), [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview).

## 2. Install and connect

Run the installation cell once, then restart the kernel and continue below it. Use Python 3.10+ in your course environment.

The tools run locally. Running the agent sends requests to OpenAI and uses your API quota. Enter your API key in the hidden prompt; keep it out of notebook source.

In [ ]:
%pip install -q "langchain>=1,<2" "langgraph>=1,<2" "langchain-openai>=1,<2"

In [ ]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key (hidden): ").strip()

# Use a tool-capable model available to your account.
model = ChatOpenAI(
    model=os.environ.get("OPENAI_MODEL", "gpt-4o"),
    temperature=0,
    timeout=30,
    max_retries=1,
    default_headers={"Accept-Encoding": "gzip, deflate"},
)

## 3. Give the agent two tools

A tool is a Python function the model can request. `@tool`, type hints, and the docstring tell the framework what the function does and what inputs it needs. **Python executes the function; the model chooses when to request it.**

For a reliable classroom demo, the distance tool uses a small table of approximate city-center coordinates. Only the listed cities are supported. It calculates **great-circle distance in kilometers**, not road distance or travel time. You do not need a geocoding API or web search.

In [ ]:
import json
import math
from langchain_core.tools import tool

# Approximate latitude/longitude in degrees; classroom reference points.
CITY_COORDS = {
    "Philadelphia": (39.9526, -75.1652),
    "New York": (40.7128, -74.0060),
    "Boston": (42.3601, -71.0589),
    "Washington, DC": (38.9072, -77.0369),
    "Los Angeles": (34.0522, -118.2437),
}

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers. Use for sums, including adding distance values."""
    return a + b

@tool
def city_distance(city1: str, city2: str) -> dict:
    """Estimate great-circle distance in kilometers between two supported US cities.
    Use exact city names: Philadelphia, New York, Boston, Washington, DC,
    or Los Angeles. Not driving distance or travel time.
    """
    missing = [city for city in (city1, city2) if city not in CITY_COORDS]
    if missing:
        return {"error": f"Unsupported city: {missing}. Supported: {list(CITY_COORDS)}"}
    lat1, lon1 = map(math.radians, CITY_COORDS[city1])
    lat2, lon2 = map(math.radians, CITY_COORDS[city2])
    a = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    km = 2 * 6371.0 * math.asin(math.sqrt(max(0.0, min(1.0, a))))
    return {"from": city1, "to": city2, "distance_km": round(km, 1),
            "method": "great-circle; approximate classroom city centers"}

TOOLS = [add_numbers, city_distance]
print("Supported cities:", list(CITY_COORDS))

In [ ]:
# Test both tools directly. These calls do not use the model.
print(add_numbers.invoke({"a": 125, "b": 75}))
print(city_distance.invoke({"city1": "Philadelphia", "city2": "New York"}))
# Expected: 200 and about 130 km.

## 4. Build the agent

These few lines replace the manual tool schemas, tool dispatcher, and loop from the first notebook.

- `model`: which model to ask.
- `tools`: which functions it can request.
- `system_prompt`: how it should behave.

The **state** is the conversation: questions, tool requests, tool results, and answers. LangGraph updates that state as the agent runs.

In [ ]:
from langchain.agents import create_agent

INSTRUCTIONS = """You are a helpful urban geography teaching assistant.
Use city_distance for city distances and add_numbers for addition.
Use the exact supported city names in tool arguments.
Report distances in kilometers and explain that they are approximate great-circle distances.
If a city is unsupported, explain the limitation and ask for a supported city.
Do not invent distances or driving times. Keep answers short.
"""

agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt=INSTRUCTIONS,
)

## 5. Ask a question and look inside

We provide a question. The framework manages the model → tool → model loop.

In [ ]:
question = "How far apart are Philadelphia and New York?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"recursion_limit": 10},
)
print(result["messages"][-1].content)

In [ ]:
# See the question, model's tool request, Python result, and final answer.
for message in result["messages"]:
    message.pretty_print()

**What should you see?** A `city_distance` request, a result around **130 km**, and a final answer using that result. A tool request usually needs two model calls: one to choose the tool, and another to answer after seeing its output.

The framework ran the same loop as your scratch agent. You did not need to write that loop again. `recursion_limit` limits graph steps if the agent keeps looping; it is not a dollar budget.

**Try it:** Change the question above to “What is 125 + 75?”, “How far is Philadelphia from Boston?”, or “Say hello.” Which tool does the agent use? Does a greeting need a tool?

## 6. Make the agent more useful: add memory

Suppose a student says, “My starting city is Philadelphia,” then asks, “How far is it from Boston?” The agent needs the earlier message.

With the framework, add a **checkpointer** to save the conversation and a **thread ID** to identify it. Reuse the same ID for follow-up questions. A new ID starts a separate conversation.

This is a concrete step toward a more complete assistant: the tools stay the same, but now students can ask follow-up questions.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

chat_agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver(),
)
conversation = {"configurable": {"thread_id": "student-1"}, "recursion_limit": 10}

In [ ]:
first = chat_agent.invoke(
    {"messages": [{"role": "user", "content": "My starting city is Philadelphia."}]},
    config=conversation,
)
print(first["messages"][-1].content)

follow_up = chat_agent.invoke(
    {"messages": [{"role": "user", "content": "How far is my starting city from Boston?"}]},
    config=conversation,
)
print(follow_up["messages"][-1].content)

`InMemorySaver` stores the conversation in RAM. Restarting the kernel, or recreating `chat_agent` above with a new saver, clears it. A database checkpointer can preserve conversations across restarts. This memory does not train the model.

**Try it:** Change the thread ID to `student-2` and ask the follow-up without first giving a starting city. The agent should ask which city you mean. Inspect the messages to check what happened.

Source: [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence).

## 7. From this demo to a more complete agent

Build one feature at a time:

| Next feature | What you add | Why it helps |
| --- | --- | --- |
| More capabilities | More tested Python tools in `TOOLS`; update instructions and rebuild the agent. | The same agent can handle more tasks. |
| Reliable city lookup | A geocoding tool with clear error messages. | Supports cities beyond our small table. |
| Evidence for answers | A tool that retrieves official Philadelphia planning documents. | Answers can refer to source material. |
| More workflow control | A custom LangGraph with a validation or human-review step. | You decide exactly where checks happen. |

A future workflow could be `question → retrieve city data → check the data → answer`. LangGraph lets you represent each step as a **node** and the connections as **edges**. You do not need to write a custom graph for today's standard tool loop.

Always check whether the agent selected the right tool and used its result correctly. More features do not automatically mean better answers.

**Small challenge:** Add a `kilometers_to_miles(km: float)` tool. Test it directly, add it to `TOOLS`, update the instructions, and recreate the agent. Ask for Philadelphia–Boston distance in both units.

## 8. Where to go next

1. **Understand the loop:** revisit [agent_from_scratch.ipynb](agent_from_scratch.ipynb) and compare its manual loop with `create_agent` here.
2. **Practice tools:** read [LangChain tools](https://docs.langchain.com/oss/python/langchain/tools) and add one tool of your own.
3. **Learn graph control:** read the [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart) when you need custom steps or branches.
4. **Build a larger project:** add [conversation persistence](https://docs.langchain.com/oss/python/langgraph/persistence), then try retrieving official city documents and checking source support.

Other options to explore later: [Pi](https://github.com/earendil-works/pi) for an extensible coding harness and TypeScript agent toolkit; [Pydantic AI](https://ai.pydantic.dev/) for agents with typed outputs; and the [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents/sdk) for agents with tools, handoffs, and tracing. You do not need these additional frameworks for this lab.

**Before you finish:** Explain what code the framework replaced, what code you still had to write, and how adding memory made the agent more useful.

If imports fail, install in this kernel and restart. If a tool change seems ignored, rerun its definition and recreate the agent. If a model request fails, check your API key, model access, and internet connection.